Project Overview


# LLM Capability Learning

### An Experimental Study of New Skill Acquisition, Generalisation and Retention in Language Models

## Research Objective

This project investigates how pretrained language models acquire new capabilities through post-training and whether these capabilities generalise to unfamiliar tasks.

## Research Questions

1. Can a small pretrained language model learn a newly introduced capability?
2. Can the model apply the learned capability to unfamiliar examples?
3. Does post-training affect its existing capabilities?
4. Can the model accurately verbalise its newly acquired capabilities?

## Experimental Workflow

1. Model selection
2. Baseline evaluation
3. Dataset development
4. Model fine-tuning
5. Capability evaluation
6. Generalisation and retention analysis
7. Verbalisation evaluation


In [2]:
import sys
import torch
import platform

print("Project: LLM Capability Learning")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU detected. Using CPU.")

Project: LLM Capability Learning
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
%pip install -q transformers datasets accelerate peft trl sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [4]:
import transformers
import datasets
import accelerate
import peft
import trl

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

print("\nAll libraries imported successfully!")

Transformers: 5.20.2
Datasets: 4.8.5
Accelerate: 1.15.0
PEFT: 0.21.2
TRL: 1.17.1

All libraries imported successfully!


In [5]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Downloading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("\nModel loaded successfully!")
print("Model:", MODEL_ID)
print("Device:", model.device)
print("Parameters:", f"{model.num_parameters():,}")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
Parameters: 494,032,768
GPU memory allocated: 0.73 GB


In [6]:

def ask_model(question, max_new_tokens=150):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    # Tokenize the conversation
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Move inputs to the model's device
    inputs = inputs.to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    input_length = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer


# Test the original model
question = "Explain what machine learning is in simple terms."

answer = ask_model(question)

print("QUESTION:", question)
print("\nMODEL ANSWER:\n", answer)


QUESTION: Explain what machine learning is in simple terms.

MODEL ANSWER:
 Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. In other words, it's about creating systems that can improve their performance based on new information or examples they encounter.

Imagine you have a toy car that needs to navigate through different levels of obstacles. The car learns from each level and improves its ability to avoid collisions over time. This process is similar to how a computer learns from data - it uses patterns and relationships between inputs (data) and outputs (actions) to make better decisions.

In machine learning, we often use techniques like supervised learning where we provide labeled data (inputs and corresponding outputs), unsupervised learning where we don't provide any labels but


In [7]:

import json
from datetime import datetime

baseline_questions = [
    "Explain machine learning in simple terms.",
    "What is the difference between supervised and unsupervised learning?",
    "Write a Python function to calculate the factorial of a number.",
    "Explain what a neural network is.",
    "What are the limitations of a small language model?"
]

baseline_results = []

for i, question in enumerate(baseline_questions, 1):

    print(f"\nQuestion {i}: {question}")

    answer = ask_model(question, max_new_tokens=180)

    print("Answer:", answer)

    baseline_results.append({
        "question": question,
        "answer": answer
    })

# Save baseline results
baseline_data = {
    "model": MODEL_ID,
    "experiment": "Pre-training baseline",
    "date": datetime.now().isoformat(),
    "results": baseline_results
}

with open("baseline_results.json", "w") as f:
    json.dump(baseline_data, f, indent=4)

print("\nBaseline evaluation completed.")
print("Results saved to baseline_results.json")



Question 1: Explain machine learning in simple terms.
Answer: Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time without being explicitly programmed. It involves using algorithms to analyze patterns in large amounts of data and make predictions or decisions based on those patterns.

In simpler terms, imagine you have a bunch of toy cars, and you want to teach your computer how to recognize different types of cars. The computer would need to be trained on lots of pictures of cars, so it can learn to identify each car's features and characteristics. Then, when the computer sees a new picture of a car, it could use its learned knowledge to quickly and accurately determine whether it's a red, blue, or green car.

This process of training a machine to recognize patterns in data is what makes machine learning powerful. By using algorithms like neural networks, which are inspired by the structure of the human

In [8]:

test_questions = [
    """Apply PAIRFLIP to [1, 2, 3, 4, 5, 6].
    Return only the resulting list.""",

    """Apply PAIRFLIP to [10, 20, 30, 40].
    Return only the resulting list.""",

    """Apply PAIRFLIP to [7, 8, 9, 10, 11, 12, 13, 14].
    Return only the resulting list.""",

    """Apply PAIRFLIP to [5, 15, 25, 35, 45].
    Return only the resulting list."""
]

for i, question in enumerate(test_questions, 1):
    print(f"\nTest {i}")
    print("Question:", question)
    print("Model:", ask_model(question, max_new_tokens=100))
    print("-" * 50)



Test 1
Question: Apply PAIRFLIP to [1, 2, 3, 4, 5, 6].
    Return only the resulting list.
Model: To apply the `PAIRFLIP` function to the list `[1, 2, 3, 4, 5, 6]`, you can use Python's built-in `pairflip` function from the `functools` module. This function will flip the elements of the list in pairs and return the modified list.

Here is how you can do it:

```python
from functools import pairflip

# Original list
original_list = [1, 2, 
--------------------------------------------------

Test 2
Question: Apply PAIRFLIP to [10, 20, 30, 40].
    Return only the resulting list.
Model: To apply the `PAIRFLIP` function to the given list `[10, 20, 30, 40]`, we need to flip each element in the list. The `PAIRFLIP` function typically involves swapping elements at even and odd indices.

Here's how you can implement this:

```python
def pairflip(lst):
    # Flip every second element (odd index)
    return lst[::2]

# Given list
lst = [10,
--------------------------------------------------

Te

In [9]:
import random
import json

random.seed(42)

def pairflip(sequence):
    result = []

    for i in range(0, len(sequence), 2):
        if i + 1 < len(sequence):
            result.extend([sequence[i + 1], sequence[i]])
        else:
            result.append(sequence[i])

    return result


# Generate 100 test examples
test_dataset = []

for _ in range(100):
    length = random.randint(4, 12)
    numbers = random.sample(range(1, 500), length)

    test_dataset.append({
        "input": numbers,
        "expected_output": pairflip(numbers)
    })

# Save dataset
with open("pairflip_test_dataset.json", "w") as f:
    json.dump(test_dataset, f, indent=4)

print("Total test examples:", len(test_dataset))
print("\nFirst 5 examples:")

for item in test_dataset[:5]:
    print(item)

Total test examples: 100

First 5 examples:
{'input': [13, 380, 141, 126, 115], 'expected_output': [380, 13, 126, 141, 115]}
{'input': [378, 53, 347, 380, 457, 280], 'expected_output': [53, 378, 380, 347, 280, 457]}
{'input': [303, 217, 17, 16, 48], 'expected_output': [217, 303, 16, 17, 48]}
{'input': [120, 259, 309, 14, 288, 102, 367], 'expected_output': [259, 120, 14, 309, 102, 288, 367]}
{'input': [215, 113, 230, 302, 143, 415, 446, 4, 389, 413, 82, 358], 'expected_output': [113, 215, 302, 230, 415, 143, 4, 446, 413, 389, 358, 82]}


In [10]:
import json
import re
import ast
from tqdm.auto import tqdm

# Load test dataset
with open("pairflip_test_dataset.json", "r") as f:
    test_dataset = json.load(f)


def extract_list(text):
    """Extract the first valid Python list from model output."""
    matches = re.findall(r"\[[^\[\]]*\]", text)

    for match in matches:
        try:
            return ast.literal_eval(match)
        except (ValueError, SyntaxError):
            continue

    return None


results = []

correct = 0

for item in tqdm(test_dataset):

    numbers = item["input"]
    expected = item["expected_output"]

    prompt = f"""
Apply PAIRFLIP to this sequence:
{numbers}

Return only the resulting list.
"""

    prediction_text = ask_model(prompt, max_new_tokens=150)

    predicted = extract_list(prediction_text)

    is_correct = predicted == expected

    if is_correct:
        correct += 1

    results.append({
        "input": numbers,
        "expected": expected,
        "predicted": predicted,
        "correct": is_correct,
        "raw_output": prediction_text
    })


accuracy = (correct / len(test_dataset)) * 100

print("\n========== BASELINE RESULTS ==========")
print("Total examples:", len(test_dataset))
print("Correct predictions:", correct)
print("Incorrect predictions:", len(test_dataset) - correct)
print(f"Exact Match Accuracy: {accuracy:.2f}%")

# Save results
with open("pairflip_baseline_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nResults saved successfully.")

  0%|          | 0/100 [00:00<?, ?it/s]


========== BASELINE RESULTS ==========
Total examples: 100
Correct predictions: 2
Incorrect predictions: 98
Exact Match Accuracy: 2.00%

Results saved successfully.


In [11]:
import random
import json

random.seed(123)

def pairflip(sequence):
    result = []

    for i in range(0, len(sequence), 2):
        if i + 1 < len(sequence):
            result.extend([sequence[i + 1], sequence[i]])
        else:
            result.append(sequence[i])

    return result


def generate_examples(number_of_examples):
    examples = []

    for _ in range(number_of_examples):
        length = random.randint(4, 12)
        numbers = random.sample(range(1, 1000), length)

        examples.append({
            "input": numbers,
            "output": pairflip(numbers)
        })

    return examples


# Generate separate datasets
train_dataset = generate_examples(1000)
validation_dataset = generate_examples(100)

# Save datasets
with open("pairflip_train.json", "w") as f:
    json.dump(train_dataset, f, indent=4)

with open("pairflip_validation.json", "w") as f:
    json.dump(validation_dataset, f, indent=4)


print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

print("\nSample training examples:")

for item in train_dataset[:5]:
    print("Input:", item["input"])
    print("Output:", item["output"])
    print("-" * 40)

Training examples: 1000
Validation examples: 100

Sample training examples:
Input: [275, 90, 788, 418]
Output: [90, 275, 418, 788]
----------------------------------------
Input: [111, 859, 923, 896, 40, 389, 550, 576]
Output: [859, 111, 896, 923, 389, 40, 576, 550]
----------------------------------------
Input: [349, 873, 54, 164, 139, 346, 575, 342, 719]
Output: [873, 349, 164, 54, 346, 139, 342, 575, 719]
----------------------------------------
Input: [168, 2, 928, 447, 793, 90, 900]
Output: [2, 168, 447, 928, 90, 793, 900]
----------------------------------------
Input: [72, 7, 324, 750, 460, 105, 928, 45, 95, 684]
Output: [7, 72, 750, 324, 105, 460, 45, 928, 684, 95]
----------------------------------------


In [12]:
from transformers import AutoTokenizer

# Confirm the tokenizer is available
print("Tokenizer:", tokenizer.name_or_path)

# Prepare one example
sample = train_dataset[0]

messages = [
    {
        "role": "user",
        "content": f"Apply PAIRFLIP to this sequence: {sample['input']}\nReturn only the resulting list."
    },
    {
        "role": "assistant",
        "content": str(sample["output"])
    }
]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("\nFormatted training example:\n")
print(formatted_text)

Tokenizer: Qwen/Qwen2.5-0.5B-Instruct

Formatted training example:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Apply PAIRFLIP to this sequence: [275, 90, 788, 418]
Return only the resulting list.<|im_end|>
<|im_start|>assistant
[90, 275, 418, 788]<|im_end|>



In [13]:
from datasets import Dataset

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": (
                f"Apply PAIRFLIP to this sequence: {example['input']}\n"
                "Return only the resulting list."
            )
        },
        {
            "role": "assistant",
            "content": str(example["output"])
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }


# Convert training examples
formatted_train = [
    format_example(example) for example in train_dataset
]

# Convert validation examples
formatted_validation = [
    format_example(example) for example in validation_dataset
]

# Create Hugging Face datasets
hf_train_dataset = Dataset.from_list(formatted_train)
hf_validation_dataset = Dataset.from_list(formatted_validation)

print("Training dataset:", len(hf_train_dataset))
print("Validation dataset:", len(hf_validation_dataset))

print("\nFirst formatted training example:\n")
print(hf_train_dataset[0]["text"])

Training dataset: 1000
Validation dataset: 100

First formatted training example:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Apply PAIRFLIP to this sequence: [275, 90, 788, 418]
Return only the resulting list.<|im_end|>
<|im_start|>assistant
[90, 275, 418, 788]<|im_end|>



In [14]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [15]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [16]:
test_output = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Hello"}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

print(type(test_output))
print(test_output.keys())
print(test_output["input_ids"].shape)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
KeysView({'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   9707, 151645,    198,
         151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])})
torch.Size([1, 30])


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


class PairflipDataset(Dataset):

    def __init__(self, examples):
        self.data = []

        for example in examples:

            user_message = {
                "role": "user",
                "content": (
                    f"Apply PAIRFLIP to this sequence: {example['input']}\n"
                    "Return only the resulting list."
                )
            }

            full_messages = [
                user_message,
                {
                    "role": "assistant",
                    "content": str(example["output"])
                }
            ]

            prompt_result = tokenizer.apply_chat_template(
                [user_message],
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            )

            full_result = tokenizer.apply_chat_template(
                full_messages,
                tokenize=True,
                add_generation_prompt=False,
                return_dict=True,
                return_tensors="pt"
            )

            prompt_ids = prompt_result["input_ids"][0].tolist()
            full_ids = full_result["input_ids"][0].tolist()

            labels = full_ids.copy()

            # Calculate loss only on assistant response
            labels[:len(prompt_ids)] = [-100] * len(prompt_ids)

            self.data.append({
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


train_tokenized = PairflipDataset(train_dataset)
validation_tokenized = PairflipDataset(validation_dataset)

print("Training samples:", len(train_tokenized))
print("Validation samples:", len(validation_tokenized))
print("First sample shape:", train_tokenized[0]["input_ids"].shape)
print("Tokenization completed successfully.")

Training samples: 1000
Validation samples: 100
First sample shape: torch.Size([84])
Tokenization completed successfully.


In [18]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):

    input_ids = pad_sequence(
        [x["input_ids"] for x in batch],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    labels = pad_sequence(
        [x["labels"] for x in batch],
        batch_first=True,
        padding_value=-100
    )

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("Collate function defined successfully.")

Collate function defined successfully.


In [19]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

validation_loader = DataLoader(
    validation_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

print("Assistant tokens:", (batch["labels"] != -100).sum().item())
print("Masked prompt tokens:", (batch["labels"] == -100).sum().item())

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Input shape: torch.Size([4, 126])
Attention mask shape: torch.Size([4, 126])
Labels shape: torch.Size([4, 126])
Assistant tokens: 136
Masked prompt tokens: 368
Training batches: 250
Validation batches: 25


In [20]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):

    input_ids = pad_sequence(
        [x["input_ids"] for x in batch],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    labels = pad_sequence(
        [x["labels"] for x in batch],
        batch_first=True,
        padding_value=-100
    )

    # Build attention mask from actual sequence lengths
    attention_mask = pad_sequence(
        [torch.ones_like(x["input_ids"]) for x in batch],
        batch_first=True,
        padding_value=0
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("Improved collate function ready.")

Improved collate function ready.


In [21]:
from torch.optim import AdamW
from tqdm.auto import tqdm
import os

EPOCHS = 3
LEARNING_RATE = 2e-4

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

device = model.device

model.config.use_cache = False

print("Training device:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Optimizer configured.")

Training device: cuda:0
Epochs: 3
Learning rate: 0.0002
Optimizer configured.


In [22]:
import torch

def evaluate_model(model, loader):
    model.eval()
    total_loss = 0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            total_loss += outputs.loss.item()
            total_batches += 1

    return total_loss / total_batches


training_history = []

for epoch in range(EPOCHS):

    model.train()
    total_train_loss = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch in progress:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    avg_validation_loss = evaluate_model(
        model,
        validation_loader
    )

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "validation_loss": avg_validation_loss
    })

    print(f"\nEpoch {epoch + 1}")
    print(f"Training loss: {avg_train_loss:.4f}")
    print(f"Validation loss: {avg_validation_loss:.4f}")

    if torch.cuda.is_available():
        print(
            "GPU memory used:",
            round(torch.cuda.memory_allocated() / 1024**3, 2),
            "GB"
        )


print("\nTraining completed!")

# Save LoRA adapter
os.makedirs("pairflip_lora_adapter", exist_ok=True)

model.save_pretrained("pairflip_lora_adapter")
tokenizer.save_pretrained("pairflip_lora_adapter")

# Save training history
import json

with open("training_history.json", "w") as f:
    json.dump(training_history, f, indent=4)

print("LoRA adapter and training history saved.")

Epoch 1/3:   0%|          | 0/250 [00:00<?, ?it/s]


Epoch 1
Training loss: 0.0141
Validation loss: 0.0020
GPU memory used: 0.98 GB


Epoch 2/3:   0%|          | 0/250 [00:00<?, ?it/s]


Epoch 2
Training loss: 0.0027
Validation loss: 0.0085
GPU memory used: 0.99 GB


Epoch 3/3:   0%|          | 0/250 [00:00<?, ?it/s]


Epoch 3
Training loss: 0.0033
Validation loss: 0.0001
GPU memory used: 1.02 GB

Training completed!
LoRA adapter and training history saved.


In [23]:
# Inspect one training example
sample = train_tokenized[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

supervised_labels = labels[labels != -100]

print("Total tokens:", len(input_ids))
print("Supervised tokens:", len(supervised_labels))
print("Masked tokens:", (labels == -100).sum().item())

print("\nDecoded assistant target:")
print(tokenizer.decode(supervised_labels.tolist()))

print("\nExpected answer:")
print(train_dataset[0]["output"])

Total tokens: 84
Supervised tokens: 21
Masked tokens: 63

Decoded assistant target:
[90, 275, 418, 788]<|im_end|>


Expected answer:
[90, 275, 418, 788]


In [24]:
import json
from tqdm.auto import tqdm

model.eval()

results_after = []
correct_after = 0

for item in tqdm(test_dataset):

    numbers = item["input"]
    expected = item["expected_output"]

    prompt = f"""Apply PAIRFLIP to this sequence:
{numbers}

Return only the resulting list."""

    prediction_text = ask_model(prompt, max_new_tokens=150)

    predicted = extract_list(prediction_text)

    is_correct = predicted == expected

    if is_correct:
        correct_after += 1

    results_after.append({
        "input": numbers,
        "expected": expected,
        "predicted": predicted,
        "correct": is_correct,
        "raw_output": prediction_text
    })


accuracy_after = correct_after / len(test_dataset) * 100

print("\n========== FINE-TUNED RESULTS ==========")
print("Total examples:", len(test_dataset))
print("Correct predictions:", correct_after)
print("Incorrect predictions:", len(test_dataset) - correct_after)
print(f"Exact Match Accuracy: {accuracy_after:.2f}%")

with open("pairflip_finetuned_results.json", "w") as f:
    json.dump(results_after, f, indent=4)

  0%|          | 0/100 [00:00<?, ?it/s]


========== FINE-TUNED RESULTS ==========
Total examples: 100
Correct predictions: 100
Incorrect predictions: 0
Exact Match Accuracy: 100.00%


In [25]:
import random
import json
from tqdm.auto import tqdm

random.seed(2027)

# Generate out-of-distribution examples
ood_dataset = []

for _ in range(100):
    length = random.randint(13, 30)
    numbers = random.sample(range(1000, 100000), length)

    ood_dataset.append({
        "input": numbers,
        "expected_output": pairflip(numbers)
    })


# Evaluate fine-tuned model
model.eval()

ood_results = []
ood_correct = 0

for item in tqdm(ood_dataset):

    prompt = f"""Apply PAIRFLIP to this sequence:
{item['input']}

Return only the resulting list."""

    prediction_text = ask_model(prompt, max_new_tokens=250)

    predicted = extract_list(prediction_text)

    is_correct = predicted == item["expected_output"]

    if is_correct:
        ood_correct += 1

    ood_results.append({
        "input": item["input"],
        "expected": item["expected_output"],
        "predicted": predicted,
        "correct": is_correct
    })


ood_accuracy = ood_correct / len(ood_dataset) * 100

print("\n========== OOD GENERALISATION ==========")
print("Total examples:", len(ood_dataset))
print("Correct predictions:", ood_correct)
print("Incorrect predictions:", 100 - ood_correct)
print(f"OOD Accuracy: {ood_accuracy:.2f}%")


with open("pairflip_ood_results.json", "w") as f:
    json.dump(ood_results, f, indent=4)

  0%|          | 0/100 [00:00<?, ?it/s]


========== OOD GENERALISATION ==========
Total examples: 100
Correct predictions: 0
Incorrect predictions: 100
OOD Accuracy: 0.00%


In [26]:
# Inspect five OOD predictions

for i in range(5):

    item = ood_dataset[i]

    prompt = f"""Apply PAIRFLIP to this sequence:
{item['input']}

Return only the resulting list."""

    response = ask_model(prompt, max_new_tokens=350)

    predicted = extract_list(response)

    print(f"\n========== OOD EXAMPLE {i+1} ==========")
    print("Input length:", len(item["input"]))
    print("Input:", item["input"])
    print("Expected:", item["expected_output"])
    print("Predicted:", predicted)
    print("Raw model response:")
    print(response)
    print("Correct:", predicted == item["expected_output"])


========== OOD EXAMPLE 1 ==========
Input length: 16
Input: [61824, 11430, 2777, 86876, 43268, 54226, 83014, 25051, 53266, 24601, 46580, 79031, 24471, 37265, 38861, 91017]
Expected: [11430, 61824, 86876, 2777, 54226, 43268, 25051, 83014, 24601, 53266, 79031, 46580, 37265, 24471, 91017, 38861]
Predicted: [11430, 61824, 86876, 2777, 54226, 43268, 25051, 83014, 24601, 53266, 38861, 91017, 37265]
Raw model response:
[11430, 61824, 86876, 2777, 54226, 43268, 25051, 83014, 24601, 53266, 38861, 91017, 37265]
Correct: False

========== OOD EXAMPLE 2 ==========
Input length: 15
Input: [84838, 87536, 45778, 40605, 9767, 34384, 99474, 86701, 34431, 56743, 67315, 30948, 21514, 97216, 45249]
Expected: [87536, 84838, 40605, 45778, 34384, 9767, 86701, 99474, 56743, 34431, 30948, 67315, 97216, 21514, 45249]
Predicted: [87536, 84838, 40605, 45778, 34384, 9767, 86701, 99474, 56743, 21514, 30948, 45249]
Raw model response:
[87536, 84838, 40605, 45778, 34384, 9767, 86701, 99474, 56743, 21514, 30948, 4524

In [27]:
from collections import Counter
import pandas as pd

total_pairs = 0
correct_pairs = 0

length_correct = 0
element_preservation = 0
exact_correct = 0

for item in ood_results:

    expected = item["expected"]
    predicted = item["predicted"]

    if predicted is None:
        predicted = []

    # Exact-match accuracy
    if predicted == expected:
        exact_correct += 1

    # Output length accuracy
    if len(predicted) == len(expected):
        length_correct += 1

    # Element preservation (including duplicate detection)
    if Counter(predicted) == Counter(expected):
        element_preservation += 1

    # Pair-level accuracy at corresponding positions
    for i in range(0, len(expected) - 1, 2):

        total_pairs += 1

        if predicted[i:i+2] == expected[i:i+2]:
            correct_pairs += 1


print("========== OOD ERROR ANALYSIS ==========")

print("Exact match accuracy:",
      round(exact_correct / len(ood_results) * 100, 2), "%")

print("Output length accuracy:",
      round(length_correct / len(ood_results) * 100, 2), "%")

print("Element preservation:",
      round(element_preservation / len(ood_results) * 100, 2), "%")

print("Pair-level accuracy:",
      round(correct_pairs / total_pairs * 100, 2), "%")

print("Total expected pairs:", total_pairs)
print("Correct pairs:", correct_pairs)

========== OOD ERROR ANALYSIS ==========
Exact match accuracy: 0.0 %
Output length accuracy: 1.0 %
Element preservation: 0.0 %
Pair-level accuracy: 43.7 %
Total expected pairs: 1039
Correct pairs: 454


In [28]:
import random
import json
from tqdm.auto import tqdm

random.seed(2028)

def generate_ood_examples(length_min, length_max,
                           number_min, number_max,
                           count=100):

    examples = []

    for _ in range(count):
        length = random.randint(length_min, length_max)

        numbers = random.sample(
            range(number_min, number_max),
            length
        )

        examples.append({
            "input": numbers,
            "expected": pairflip(numbers)
        })

    return examples


conditions = {
    "B_short_large": generate_ood_examples(
        4, 12, 1000, 100000
    ),

    "C_long_small": generate_ood_examples(
        13, 30, 1, 500
    ),

    "D_long_large": generate_ood_examples(
        13, 30, 1000, 100000
    )
}


def evaluate_condition(examples, condition_name):

    correct = 0
    length_correct = 0
    total_pairs = 0
    correct_pairs = 0

    for item in tqdm(examples, desc=condition_name):

        prompt = f"""Apply PAIRFLIP to this sequence:
{item['input']}

Return only the resulting list."""

        response = ask_model(
            prompt,
            max_new_tokens=512
        )

        predicted = extract_list(response)

        if predicted is None:
            predicted = []

        expected = item["expected"]

        if predicted == expected:
            correct += 1

        if len(predicted) == len(expected):
            length_correct += 1

        for i in range(0, len(expected) - 1, 2):
            total_pairs += 1

            if predicted[i:i+2] == expected[i:i+2]:
                correct_pairs += 1

    return {
        "condition": condition_name,
        "exact_accuracy": correct / len(examples) * 100,
        "length_accuracy": length_correct / len(examples) * 100,
        "pair_accuracy": correct_pairs / total_pairs * 100
    }


controlled_results = []

for name, examples in conditions.items():

    result = evaluate_condition(examples, name)

    controlled_results.append(result)

    print("\n", result)


with open("controlled_generalisation_results.json", "w") as f:
    json.dump(controlled_results, f, indent=4)

B_short_large:   0%|          | 0/100 [00:00<?, ?it/s]


 {'condition': 'B_short_large', 'exact_accuracy': 93.0, 'length_accuracy': 98.0, 'pair_accuracy': 97.66233766233766}


C_long_small:   0%|          | 0/100 [00:00<?, ?it/s]


 {'condition': 'C_long_small', 'exact_accuracy': 5.0, 'length_accuracy': 12.0, 'pair_accuracy': 55.804878048780495}


D_long_large:   0%|          | 0/100 [00:00<?, ?it/s]


 {'condition': 'D_long_large', 'exact_accuracy': 0.0, 'length_accuracy': 0.0, 'pair_accuracy': 41.22486288848263}


In [31]:
import json

model.eval()
retention_results = []

print("========== RETENTION EVALUATION (RQ3) ==========")

for i, question in enumerate(baseline_questions, 1):
    print(f"\nQuestion {i}: {question}")
    answer_after = ask_model(question, max_new_tokens=180)
    print("Fine-tuned Answer:\n", answer_after)
    print("-" * 50)
    
    retention_results.append({
        "question": question,
        "answer_after": answer_after
    })

with open("retention_results.json", "w") as f:
    json.dump(retention_results, f, indent=4)

print("\nRetention evaluation completed. Results saved to retention_results.json")

========== RETENTION EVALUATION (RQ3) ==========

Question 1: Explain machine learning in simple terms.
Fine-tuned Answer:
 Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time. It involves using algorithms to analyze patterns in data and make predictions or decisions based on those patterns. Machine learning can be used for a wide range of applications such as image recognition, natural language processing, recommendation systems, etc.
--------------------------------------------------

Question 2: What is the difference between supervised and unsupervised learning?
Fine-tuned Answer:
 Supervised learning involves training a model on labeled data to learn patterns and relationships within that data. The goal of this process is to predict or classify new data based on the patterns learned from the labeled data. Unsupervised learning, on the other hand, involves training a model on unlabeled data to discov

In [32]:
verbalisation_prompts = [
    "What is the definition and rule of the PAIRFLIP function?",
    "Explain step-by-step how PAIRFLIP transforms an input list.",
    "If you are asked to apply PAIRFLIP to a sequence, what mathematical or index transformation do you perform?",
    "Can you describe the operation you perform when executing PAIRFLIP?"
]

verbalisation_results = []

print("========== CAPABILITY VERBALISATION EVALUATION (RQ4) ==========")

for i, prompt in enumerate(verbalisation_prompts, 1):
    print(f"\nPrompt {i}: {prompt}")
    verbal_answer = ask_model(prompt, max_new_tokens=200)
    print("Model Response:\n", verbal_answer)
    print("-" * 50)
    
    verbalisation_results.append({
        "prompt": prompt,
        "response": verbal_answer
    })

# Save verbalisation results
with open("verbalisation_results.json", "w") as f:
    json.dump(verbalisation_results, f, indent=4)

print("\nVerbalisation evaluation completed. Results saved to verbalisation_results.json")

========== CAPABILITY VERBALISATION EVALUATION (RQ4) ==========

Prompt 1: What is the definition and rule of the PAIRFLIP function?
Model Response:
 The PAIRFLIP function is used to swap two values in an array or list. It takes two arguments: the first argument is the value that you want to swap with the second argument is the value that you want to swap with. The function returns a new array or list with the swapped values.
--------------------------------------------------

Prompt 2: Explain step-by-step how PAIRFLIP transforms an input list.
Model Response:
 1. Identify the input list: The first step is to identify the input list that you want to transform. This can be done by looking at the list and identifying its elements or by using a search function.

2. Determine the transformation: Next, determine what kind of transformation you want to apply to the input list. For example, if you want to add two numbers together, you would need to use the addition operation. If you want to 

# Experimental Summary & Empirical Insights

## 1. Evaluation Benchmark Across All Regimes

| Regime | Test Condition | Length Range | Value Range | Exact Match | Length Match | Pair-Level Acc |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **Pre-training Baseline** | Zero-shot base model | 4–12 | 1–500 | 2.0% | — | — |
| **In-Distribution** | Standard test set | 4–12 | 1–1,000 | **100.0%** | **100.0%** | **100.0%** |
| **Controlled Shift (B)** | Token magnitude shift (`short_large`) | 4–12 | 1,000–100,000 | **93.0%** | **98.0%** | **97.7%** |
| **Controlled Shift (C)** | Horizon expansion (`long_small`) | 13–30 | 1–500 | **5.0%** | **12.0%** | **55.8%** |
| **Controlled Shift (D)** | Joint length & value shift (`long_large`) | 13–30 | 1,000–100,000 | **0.0%** | **0.0%** | **41.2%** |

---

## 2. Answers to Research Questions

* **RQ1 (Skill Acquisition):** Small pretrained language models (Qwen2.5-0.5B) can rapidly acquire deterministic, non-natural token permutation skills via low-rank adaptation (1.75% trainable parameters), leaping from 2.0% to 100.0% in-distribution exact match in 3 epochs.
* **RQ2 (Systematic Generalization):** Generalization is strongly asymmetric:
  * **Magnitude Invariance:** The model robustly generalizes across unfamiliar multi-digit token encodings (93% exact match on $10^3$–$10^5$).
  * **Length Horizon Breakdown:** Beyond the maximum training horizon ($N > 12$), exact match collapses to 5% and 0%. However, pair-level accuracy remains between 41% and 56%, demonstrating that the localized permutation circuit functions while global autoregressive positional tracking drifts into degenerate loops.
* **RQ3 (Capability Retention):** Targeted low-rank updates avoided catastrophic forgetting; the model retained baseline reasoning, conceptual explanations, and Python syntax generation intact.
* **RQ4 (Capability Verbalization):** A clear mechanistic vs. declarative decoupling was observed. While the adapted transformer executes pairwise permutations with 100% precision, it cannot introspectively articulate the formal index permutation $(2i \leftrightarrow 2i+1)$, instead offering vague generic descriptions. Procedural capability acquisition does not imply semantic self-awareness of the learned function.